# About this Notebook


Automated Finance Research Report + Trading Signal Extraction
with LangGraph + OpenAI + Exa + yfinance

New in this version
- Technical indicator node with SMA, RSI, MACD
- Exa content grounding with title, url, excerpt
- TradingSignal schema with direction, horizon, conviction, entries
- Signal scoring and JSON sidecar for downstream use

__Setup__
export OPENAI_API_KEY=... EXA_API_KEY=...


# Dependencies

In [1]:
!pip install -q langchain==0.3.27 \
                langgraph==0.6.7 \
                langchain-openai==0.3.32 \
                langchain_experimental==0.3.4 \
                langchain_community==0.3.29 \
                python-dotenv==1.0.1 \
                langchain-core==0.3.75 \
                serpapi==0.1.5 \
                google-search-results==2.4.2 \
                exa_py==1.7.0

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 11.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency confli

# Imports for API

In [2]:
from dotenv import load_dotenv
import os

# Imports

In [ ]:
from __future__ import annotations
import os
import json
from typing import TypedDict, List, Dict, Any
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import yfinance as yf
from exa_py import Exa
from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END


# Setup Env

In [ ]:
load_dotenv()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
# Force function_calling pathway, never json_schema
llm = ChatOpenAI(model=MODEL, temperature=0.1)
exa = Exa(os.getenv("EXA_API_KEY"))

# Helpers

In [ ]:
def _ensure_single_symbol_df(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    """Normalize yfinance frames that sometimes come with MultiIndex columns."""
    if isinstance(df.columns, pd.MultiIndex):
        try:
            df = df.xs(ticker, axis=1, level=1)
        except Exception:
            try:
                df = df.droplevel(1, axis=1)
            except Exception:
                pass
    return df


# Setup Graphstate

In [ ]:
class NewsDoc(BaseModel):
    title: str
    url: str
    excerpt: str

class ResearchState(TypedDict):
    ticker: str
    date: str
    price_data: List[Dict[str, Any]]
    indicators: Dict[str, Any]
    news: List[Dict[str, str]]  # NewsDoc dicts
    headlines: List[str]
    bullets: Dict[str, Any]
    draft_md: str
    critique: Dict[str, Any]
    report_md: str
    trading_signals: List[Dict[str, Any]]
    iter: int

# Structured Models

In [ ]:
class ExtractedSignals(BaseModel):
    trend_summary: str
    drivers: List[str]
    risks: List[str]
    data_highlights: List[str]

class Critique(BaseModel):
    needs_revision: bool
    strengths: List[str] = []
    suggestions: List[str] = []
    risk_flags: List[str] = []

class TradingSignal(BaseModel):
    direction: str = Field(..., description="long or short")
    horizon: str = Field(..., description="intraday, swing, position")
    conviction: float = Field(..., ge=0, le=1, description="0 to 1 confidence")
    entry: float
    stop: float
    target: float

    rationale: str
    evidence: List[str] = Field(default_factory=list, description="urls or short quotes")
    risks: List[str] = Field(default_factory=list)

    @field_validator("direction")
    @classmethod
    def _dir(cls, v):
        v2 = v.lower().strip()
        if v2 not in {"long", "short"}:
            raise ValueError("direction must be long or short")
        return v2

class TradingSignalList(BaseModel):
    # Wrapper required for LangChain function_calling structured output
    signals: List[TradingSignal]

# Indicator Helpers


In [ ]:
def compute_indicators_df(df: pd.DataFrame) -> Dict[str, Any]:
    df = _ensure_single_symbol_df(df, "_")
    close = df["Close"].astype(float)

    # SMA
    df["sma_10"] = close.rolling(10).mean()
    df["sma_20"] = close.rolling(20).mean()

    # RSI
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / (avg_loss.replace(0, np.nan))
    df["rsi_14"] = 100 - (100 / (1 + rs))

    # MACD
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    hist = macd - signal
    df["macd"] = macd
    df["macd_signal"] = signal
    df["macd_hist"] = hist

    # simple returns
    df["ret_1d"] = close.pct_change(1)
    df["ret_5d"] = close.pct_change(5)

    last = df.dropna().iloc[-1]

    def _scalar(x):
        try:
            return float(x.iloc[0])
        except Exception:
            return float(x)

    return {
        "asof": last.name.strftime("%Y-%m-%d"),
        "close": _scalar(last["Close"]),
        "sma_10": _scalar(last["sma_10"]),
        "sma_20": _scalar(last["sma_20"]),
        "rsi_14": _scalar(last["rsi_14"]),
        "macd": _scalar(last["macd"]),
        "macd_signal": _scalar(last["macd_signal"]),
        "macd_hist": _scalar(last["macd_hist"]),
        "ret_1d": _scalar(last["ret_1d"]),
        "ret_5d": _scalar(last["ret_5d"]),
    }

# Nodes

In [ ]:
def gather_data(state: ResearchState) -> ResearchState:
    ticker = state["ticker"]
    end = datetime.today()
    start = end - timedelta(days=60)
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)

    # normalize possible MultiIndex columns from yfinance
    df = _ensure_single_symbol_df(df, ticker)

    df_pd = df.reset_index()[["Date", "Close", "Volume"]].dropna()
    price_data = [
        {"date": r.Date.strftime("%Y-%m-%d"), "close": float(r.Close), "volume": int(r.Volume)}
        for r in df_pd.itertuples(index=False)
    ]

    indicators = compute_indicators_df(df)

    # Exa search last 7 days, include contents
    date_cutoff = (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
    search_response = exa.search_and_contents(
        ticker, use_autoprompt=True, start_published_date=date_cutoff
    )
    news_docs: List[NewsDoc] = []
    for r in search_response.results[:8]:
        if not r.title or not r.url:
            continue
        text = (r.text or "").strip()
        excerpt = text[:1800]  # keep context small
        news_docs.append(NewsDoc(title=r.title, url=r.url, excerpt=excerpt))

    headlines = [nd.title for nd in news_docs[:5]]

    return {
        **state,
        "price_data": price_data,
        "indicators": indicators,
        "news": [nd.model_dump() for nd in news_docs],
        "headlines": headlines,
    }


def extract_signals(state: ResearchState) -> ResearchState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an equity research associate. Extract compact signals from the data."),
        ("human", "Ticker: {ticker}\nDate: {date}\n"
                  "Indicators:```json\n{ind}\n```\n"
                  "Prices:```json\n{prices}\n```\n"
                  "Top headlines:\n{heads}\n\n"
                  "Return JSON fields: trend_summary, drivers, risks, data_highlights."),
    ])

    chain = prompt | llm.with_structured_output(ExtractedSignals, method="function_calling")
    res = chain.invoke({
        "ticker": state["ticker"],
        "date": state["date"],
        "ind": json.dumps(state["indicators"], indent=2),
        "prices": json.dumps(state["price_data"][-7:], indent=2),
        "heads": "\n".join(state["headlines"]),
    })
    return {**state, "bullets": res.model_dump()}


def draft_report(state: ResearchState) -> ResearchState:
    sys = SystemMessage(content=(
        "You write concise sell side style research notes for pros."
        " Sections: Summary, Drivers, Technicals, Risks, Outlook, Sources."
        " Cite sources as plain urls in Sources."
    ))

    sources = "\n".join([n["url"] for n in state["news"][:5]])
    human = HumanMessage(content=(
        f"Ticker: {state['ticker']}\nDate: {state['date']}\n"
        f"Signals:```json\n{json.dumps(state['bullets'], indent=2)}\n```\n"
        f"Indicators:```json\n{json.dumps(state['indicators'], indent=2)}\n```\n"
        f"Sources:\n{sources}\n\nDraft in Markdown."
    ))
    text = (llm | StrOutputParser()).invoke([sys, human])
    return {**state, "draft_md": text, "report_md": text}


def review(state: ResearchState) -> ResearchState:
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a meticulous equity research reviewer."),
        ("human", "Review this note and return JSON critique.\nNote:\n```\n{note}\n```"),
    ])
    chain = prompt | llm.with_structured_output(Critique, method="function_calling")
    res = chain.invoke({"note": state["report_md"]})
    return {**state, "critique": res.model_dump()}


def improve(state: ResearchState) -> ResearchState:
    if not state.get("critique", {}).get("needs_revision"):
        return state
    sys = SystemMessage(content="Revise the note using reviewer suggestions. Keep it under 450 words.")
    human = HumanMessage(content=(
        "Original note:\n```\n" + state["report_md"] + "\n```\n\n"
        "Suggestions:\n" + json.dumps(state["critique"].get("suggestions", []), indent=2)
    ))
    revised = (llm | StrOutputParser()).invoke([sys, human])
    return {**state, "report_md": revised}


def extract_trading_signals(state: ResearchState) -> ResearchState:
    """LLM synthesizes a small set of structured trading signals from indicators and grounded news."""
    news_snippets = [
        {"title": n["title"], "url": n["url"], "excerpt": n["excerpt"][:400]}
        for n in state.get("news", [])[:6]
    ]
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a disciplined trading analyst. Propose at most two actionable signals."
                   " Use the indicators and grounded news. Use tight stops. Avoid overfitting."),
        ("human", "Ticker: {ticker}\nDate: {date}\n"
                   "Indicators:```json\n{ind}\n```\n"
                   "Recent news (title, url, excerpt):```json\n{news}\n```\n"
                   "Return a JSON object with field 'signals' which is a list of TradingSignal objects."),
    ])
    chain = prompt | llm.with_structured_output(TradingSignalList, method="function_calling")
    res = chain.invoke({
        "ticker": state["ticker"],
        "date": state["date"],
        "ind": json.dumps(state["indicators"], indent=2),
        "news": json.dumps(news_snippets, indent=2),
    })
    return {**state, "trading_signals": [s.model_dump() for s in res.signals]}


def score_signals(state: ResearchState) -> ResearchState:
    """Simple heuristic score to complement the LLM conviction field."""
    ind = state.get("indicators", {})
    scored = []
    for s in state.get("trading_signals", []):
        score = 0.0
        # momentum tilt
        if s["direction"] == "long":
            score += 0.5 if ind.get("macd_hist", 0) > 0 else -0.2
            score += 0.3 if ind.get("rsi_14", 50) < 70 else -0.1
        else:
            score += 0.5 if ind.get("macd_hist", 0) < 0 else -0.2
            score += 0.3 if ind.get("rsi_14", 50) > 30 else -0.1
        # distance to stop relative to target
        rr = (abs(s["target"] - s["entry"]) + 1e-6) / (abs(s["entry"] - s["stop"]) + 1e-6)
        score += 0.2 if rr >= 1.5 else -0.1
        s2 = {**s, "score": round(float(score), 3)}
        scored.append(s2)
    return {**state, "trading_signals": scored}

# Graph

In [ ]:
graph = StateGraph(ResearchState)
graph.add_node("gather", gather_data)
graph.add_node("signals", extract_signals)
graph.add_node("draft", draft_report)
graph.add_node("review", review)
graph.add_node("improve", improve)
graph.add_node("trade_signals", extract_trading_signals)
graph.add_node("score", score_signals)

# wiring
graph.add_edge(START, "gather")
graph.add_edge("gather", "signals")
graph.add_edge("signals", "draft")
graph.add_edge("draft", "review")

# always try to extract signals before finishing
def after_review(state: ResearchState) -> str:
    # do one revise pass at most, then move to trade signals
    if state.get("critique", {}).get("needs_revision") and state.get("iter", 0) < 1:
        return "improve"
    return "trade_signals"

graph.add_conditional_edges("review", after_review, {"improve": "improve", "trade_signals": "trade_signals"})
graph.add_edge("improve", "trade_signals")
graph.add_edge("trade_signals", "score")
graph.add_edge("score", END)

app = graph.compile()

# Run Application

In [8]:
if __name__ == "__main__":
    init: ResearchState = {
        "ticker": "AAPL",
        "date": datetime.today().strftime("%Y-%m-%d"),
        "price_data": [],
        "indicators": {},
        "news": [],
        "headlines": [],
        "bullets": {},
        "draft_md": "",
        "critique": {},
        "report_md": "",
        "trading_signals": [],
        "iter": 0,
    }

    from rich.console import Console
    from rich.panel import Panel
    from rich.table import Table

    console = Console()

    for event in app.stream(init):
        for node, output in event.items():
            console.rule(f"[bold]Node: {node}")
            if node == "signals":
                console.print(Panel.fit(json.dumps(output["bullets"], indent=2)))
            elif node in ("draft", "improve"):
                console.print(output["report_md"])
            elif node == "review":
                console.print(Panel.fit(json.dumps(output["critique"], indent=2)))
            elif node == "trade_signals":
                console.print(Panel.fit(json.dumps(output["trading_signals"], indent=2)))
            elif node == "score":
                # pretty table for scored signals
                tbl = Table(show_header=True, header_style="bold")
                tbl.add_column("dir")
                tbl.add_column("horizon")
                tbl.add_column("entry")
                tbl.add_column("stop")
                tbl.add_column("target")
                tbl.add_column("conviction")
                tbl.add_column("score")
                for s in output.get("trading_signals", []):
                    tbl.add_row(
                        s.get("direction", ""),
                        s.get("horizon", ""),
                        f"{s.get('entry', 0):.2f}",
                        f"{s.get('stop', 0):.2f}",
                        f"{s.get('target', 0):.2f}",
                        f"{s.get('conviction', 0):.2f}",
                        f"{s.get('score', 0):.2f}",
                    )
                console.print(tbl)

    final = app.invoke(init)
    console.rule("[bold green]Final Report")
    console.print(final["report_md"])
    console.rule("[bold green]Trading Signals JSON")
    console.print(json.dumps(final["trading_signals"], indent=2))


────────────────────────────────────────────────── Node: gather ───────────────────────────────────────────────────

────────────────────────────────────────────────── Node: signals ──────────────────────────────────────────────────

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "trend_summary": "AAPL's stock is currently trading at $234.35, showing a slight decline of 1.48% over the    │
│ last day but a positive return of 2.02% over the past five days. The stock is above its 10-day simple moving    │
│ average (SMA) of $234.44 and significantly above its 20-day SMA of $231.95, indicating a generally bullish      │
│ trend in the short term.",                                                                                      │
│   "drivers": [                                                                                                  │
│     "Recent positive momentum with a 5-day return of 2.02%.",                                                   │
│     "Strong trading volume on September 9, indicating investor interest."                                       │
│   ],                                                                                                            │
│   "risks": [                                                                                                    │
│     "Recent decline of 1.48% in the last trading session may signal short-term volatility.",                    │
│     "RSI at 55.77 suggests the stock is neither overbought nor oversold, indicating potential for further       │
│ fluctuations."                                                                                                  │
│   ],                                                                                                            │
│   "data_highlights": [                                                                                          │
│     "MACD shows a positive divergence with a value of 5.09, indicating potential upward momentum.",             │
│     "Volume on September 9 was 66,153,200, significantly higher than previous days, suggesting increased        │
│ trading activity."                                                                                              │
│   ]                                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

─────────────────────────────────────────────────── Node: draft ───────────────────────────────────────────────────

# AAPL Research Note

**Date:** September 10, 2025

## Summary
AAPL is currently trading at $234.35, reflecting a slight decline of 1.48% over the last day but a positive return 
of 2.02% over the past five days. The stock remains above its 10-day SMA of $234.44 and significantly above its 
20-day SMA of $231.95, indicating a generally bullish trend in the short term.

## Drivers
- Recent positive momentum with a 5-day return of 2.02%.
- Strong trading volume on September 9, indicating heightened investor interest.

## Technicals
- **Current Price:** $234.35
- **10-day SMA:** $234.44
- **20-day SMA:** $231.95
- **RSI (14):** 55.77 (neutral territory)
- **MACD:** 5.09 (positive divergence)
- **Volume (Sept 9):** 66,153,200 (significantly higher than previous days)

## Risks
- The recent decline of 1.48% in the last trading session may signal short-term volatility.
- RSI at 55.77 suggests the stock is neither overbought nor oversold, indicating potential for further 
fluctuations.

## Outlook
AAPL's technical indicators suggest a generally bullish trend, supported by strong trading volume and positive MACD
divergence. However, investors should remain cautious of potential short-term volatility as indicated by the recent
price decline.

## Sources
- https://www.marketwatch.com/investing/stock/aapl
- https://ca.finance.yahoo.com/quote/AAPL/
- https://www.cnn.com/markets/stocks/AAPL
- https://www.nasdaq.com/articles/apple-aapl-beats-stock-market-upswing-what-investors-need-know-0
- https://www.historicalstockprice.com/aapl-historical-stock-prices/

...

┏━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ dir   ┃ horizon  ┃ entry  ┃ stop   ┃ target ┃ conviction ┃ score ┃
┡━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ long  │ swing    │ 234.35 │ 232.00 │ 240.00 │ 0.70       │ 1.00  │
│ short │ intraday │ 234.35 │ 236.00 │ 230.00 │ 0.60       │ 0.30  │
└───────┴──────────┴────────┴────────┴────────┴────────────┴───────┘

────────────────────────────────────────────────── Final Report ───────────────────────────────────────────────────

# AAPL Research Note

**Date:** September 10, 2025

## Summary
AAPL's stock is currently experiencing a slight downtrend, closing at $234.35, down 1.48% over the last day. 
However, the stock has shown a short-term recovery with a 2.02% gain over the past five days. The upcoming launch 
of the iPhone 17 line, including the slimmer iPhone Air, is expected to drive consumer interest and sales.

## Drivers
- **iPhone 17 Launch:** The introduction of the iPhone 17 line, particularly the slimmer iPhone Air, is anticipated
to boost consumer interest and sales.
- **Market Sentiment:** AAPL has recently outperformed the broader market, contributing to positive investor 
sentiment.

## Technicals
- **Current Close:** $234.35
- **10-day SMA:** $234.44 (indicating a slight downward trend)
- **14-day RSI:** 55.77 (neutral territory)
- **MACD:** 5.09 with a histogram of 0.08, suggesting potential upward momentum.

## Risks
- **Post-Launch Volatility:** Stock price may experience volatility as initial consumer response to the iPhone 
launch is assessed.
- **Market Conditions:** Broader market trends could negatively impact AAPL's performance, particularly if market 
sentiment turns bearish.

## Outlook
While the launch of the iPhone 17 line is a significant driver for AAPL, investors should remain cautious of 
potential volatility in the stock price and broader market conditions. The technical indicators suggest a neutral 
to slightly bullish outlook in the short term.

## Sources
- [MarketWatch - AAPL](https://www.marketwatch.com/investing/stock/aapl)
- [Yahoo Finance - AAPL](https://ca.finance.yahoo.com/quote/AAPL/)
- [CNN Markets - AAPL](https://www.cnn.com/markets/stocks/AAPL)
- [Seeking Alpha - iPhone 17 
Launch](https://seekingalpha.com/news/4493392-apple-takes-wraps-off-iphone-17-line-including-slimmer-iphone-air)
- [NASDAQ - AAPL Market 
Performance](https://www.nasdaq.com/articles/apple-aapl-beats-stock-market-upswing-what-investors-need-know-0)

────────────────────────────────────────────── Trading Signals JSON ───────────────────────────────────────────────

[
  {
    "direction": "long",
    "horizon": "swing",
    "conviction": 0.7,
    "entry": 234.35,
    "stop": 232.0,
    "target": 240.0,
    "rationale": "AAPL is trading above its 20-day SMA and has a positive MACD histogram, indicating bullish 
momentum. The recent unveiling of the iPhone 17 line could drive investor sentiment positively.",
    "evidence": [
      "https://seekingalpha.com/news/4493392-apple-takes-wraps-off-iphone-17-line-including-slimmer-iphone-air"
    ],
    "risks": [
      "Market volatility due to broader economic conditions."
    ],
    "score": 1.0
  },
  {
    "direction": "short",
    "horizon": "swing",
    "conviction": 0.6,
    "entry": 234.35,
    "stop": 236.0,
    "target": 230.0,
    "rationale": "Despite recent positive news, AAPL has shown a slight decline in the last trading session and is 
near the top of its 52-week range, suggesting potential overextension.",
    "evidence": [
      "https://www.cnn.com/markets/stocks/AAPL"
    ],
    "risks": [
      "Unexpected positive news could drive the price higher."
    ],
    "score": 0.3
  }
]